In [19]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

test = pd.read_csv('test.csv')
train = pd.read_csv('train.csv')

In [2]:
train.head()

,ID,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,Footage_to_Lot_Ratio,Total_Rooms,Age_of_House,Garage_to_Footage_Ratio,Avg_Room_Size,Price,House_Orientation_Angle,Street_Alignment_Offset,Solar_Exposure_Index,Magnetic_Field_Strength,Vibration_Level
0,1,2028,2,3,1967,1.784790,2,2,1136.268444,5,58,0.000986,405.600000,11184.929934,16.722149,298.409571,235.502857,227.621575,129.770822
1,2,3519,5,3,1966,4.009947,0,10,877.567605,8,59,0.000000,439.875000,13941.315383,340.115663,43.878994,300.292055,46.684432,211.676987
2,3,4507,2,3,2014,4.122337,0,7,1093.311933,5,11,0.000000,901.400000,19686.885572,219.823215,24.542031,186.851621,10.837394,316.769266
3,4,3371,4,2,2000,1.580318,0,1,2133.114532,6,25,0.000000,561.833333,20964.530841,10.361763,147.970249,107.843644,175.620355,244.463978
4,5,2871,5,1,1974,3.426914,2,6,837.780090,6,51,0.000697,478.500000,12180.466278,329.344524,46.114469,357.571806,335.719756,135.850744


In [49]:
# T1
train['TotalArea'] = train['Square_Footage'] + train['Garage_Size'] + train['Lot_Size']
test['TotalArea'] = test['Square_Footage'] + test['Garage_Size'] + test['Lot_Size']
task1 = test['TotalArea']

In [50]:
# T2
train['Garage_to_Room_Ratio'] = train['Garage_Size']/train['Total_Rooms']
test['Garage_to_Room_Ratio'] = test['Garage_Size']/test['Total_Rooms']
task2 = test['Garage_to_Room_Ratio']

In [51]:
# T3
train['Env_Stability_Index'] = (train['Solar_Exposure_Index'] + train['Vibration_Level']) / train['Magnetic_Field_Strength']
test['Env_Stability_Index'] = (test['Solar_Exposure_Index'] + test['Vibration_Level']) / test['Magnetic_Field_Strength']
task3 = test['Env_Stability_Index']

In [52]:
# T4
sqftMean = train['Square_Footage'].mean()
train['T4'] = abs(train['Square_Footage'] - sqftMean)
sqftMean = test['Square_Footage'].mean()
test['T4'] = abs(test['Square_Footage'] - sqftMean)
task4 = test['T4']

In [57]:
#T5
x_train = train.drop(columns=['Price', 'ID'])
y_train = train['Price']
x_test = test.drop(columns=['ID'])

numCols = x_train.columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numCols)
    ]
)

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', XGBRegressor(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 500],
    'model__max_depth': [3, 6],
    'model__learning_rate': [0.01, 0.1],
    'model__subsample': [0.8, 1.0],
}

gridSearch = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
)

gridSearch.fit(x_train, y_train)
model = gridSearch.best_estimator_
predictions = model.predict(x_test)

In [58]:
rows = []
for sid, t1, t2, t3, t4, pred in zip(test['ID'], task1, task2, task3, task4, predictions):
    rows.append({'subtaskID': 1, 'datapointID': sid, 'answer': t1})
    rows.append({'subtaskID': 2, 'datapointID': sid, 'answer': t2})
    rows.append({'subtaskID': 3, 'datapointID': sid, 'answer': t3})
    rows.append({'subtaskID': 4, 'datapointID': sid, 'answer': t4})
    rows.append({'subtaskID': 5, 'datapointID': sid, 'answer': pred})

sub = pd.DataFrame(rows)
sub.to_csv('submission.csv', index=False)